1) High-level overview
2) HTTP protocol
3) Routing
4) Serialization and deserialization
5) Authentication and authorisation
6) Validation and transformation
7) Middleware
8) Requests: body, headers, query params
9) Handlers, controllers, services
10) CRUD deep dive
11) RESTful best practices
12) Databases and persistence
13) Business logic layer (BLL)
14) Caching
15) Transactional email
16) Task queues and scheduling
17) Elasticsearch
18) Error handling
19) Configuration management
20) Logging, monitoring, observability
21) Graceful shutdown
22) Security
23) Scaling and performance
24) Concurrency and parallelism
25) Object storage and large files
26) Real-time backend systems
27) Testing and code quality
28) Twelve-factor app
29) OpenAPI specification
30) Webhooks
31) DevOps for backend engineers

## 1) High-level overview

**Short notes**
- Backend = services that power clients (web, mobile, integrations): APIs, data, rules, side effects.
- Sits behind the network boundary; clients never trust it blindly, but it is the source of truth for business rules.

**Why**
- Build a mental map before frameworks: know what any backend must do (receive requests, apply rules, persist state, return responses).
- Avoid learning tools in isolation without the problems they solve.

**Where**
- Architecture docs, onboarding, designing a new service, comparing monolith vs microservices.

**How**
- Sketch request → routing → validation → business logic → data → response.
- Name the major components (API layer, domain, infrastructure) for your stack.

**Uses**
- Onboarding, system design interviews, planning features, debugging “where should this code live?”.

## 2) HTTP protocol

**Short notes**
- Application protocol for transferring resources: methods (GET, POST, PUT, PATCH, DELETE), URLs, headers, bodies, status codes.

**Why**
- Nearly all public backends speak HTTP or HTTP/2/gRPC-gateway; misunderstandings cause bugs and security issues.

**Where**
- Every REST API, BFF (backend-for-frontend), webhooks callbacks, health checks behind load balancers.

**How**
- Learn idempotency per method; request/response structure; caching headers; HTTPS/TLS termination.

**Uses**
- Designing endpoints, troubleshooting 4xx/5xx, CDN and cache behaviour, interoperability with browsers and mobiles.

## 3) Routing

**Short notes**
- Maps URL + HTTP method (+ sometimes host/version) to a handler; may include path parameters and route groups.

**Why**
- Clean URLs, separation of concerns, security (expose only intended operations).

**Where**
- Web frameworks (Express, FastAPI, Spring MVC, ASP.NET, Gin, etc.).

**How**
- Define routes with patterns (`/users/:id`), middleware chains, optional versioning (`/v1/...`).

**Uses**
- REST resources, RPC-style routes, internal admin APIs, websocket upgrade paths declared alongside HTTP routes.

## 4) Serialization and deserialization

**Short notes**
- Convert typed objects ⇄ bytes/text (JSON, Protobuf, MessagePack, CSV); deserialization is parsing untrusted input.

**Why**
- Wire format differs from memory model; failures and exploits happen at parsing boundaries.

**Where**
- Request/response bodies, message queues, event payloads, gRPC/proto, DB adapters returning rows.

**How**
- Use schema-first or explicit DTOs; strict parsers; versioning; nullable vs missing field rules.

**Uses**
- Public APIs, internal microservices contracts, exporting reports, interoperability with heterogeneous clients.

## 5) Authentication and authorisation

**Short notes**
- **Authentication:** who is this caller? (sessions, JWT, API keys, mTLS). **Authorisation:** what may they do? (RBAC, ABAC, scopes).

**Why**
- Most incidents are identity or permission mistakes; never trust client-only checks.

**Where**
- API gateways, auth middleware, service-to-service tokens, database row-level security (optional extra layer).

**How**
- Short-lived credentials, refresh flows, least privilege, audit logs, central IdP (OIDC/OAuth2) where possible.

**Uses**
- User APIs, admin panels, partner integrations, machine-to-machine jobs.

## 6) Validation and transformation

**Short notes**
- **Validation:** reject bad input (types, ranges, enums, business invariants). **Transformation:** map to domain types, normalize, trim, timezone handling.

**Why**
- Garbage in → corrupt data, security bypasses, confusing errors; transformation keeps domain code clean.

**Where**
- At the edge (controller/DTO), sometimes again in domain for invariants; never only in the database.

**How**
- Schema libraries (Pydantic, Zod, class-validator), custom validators, clear 400 responses with field errors.

**Uses**
- Form submissions, mobile payloads, bulk imports, webhook verification before side effects.

## 7) Middleware

**Short notes**
- Pluggable pipeline around handlers: runs on request and/or response (auth, logging, rate limit, CORS, tracing).

**Why**
- Cross-cutting concerns without duplicating code in every route.

**Where**
- HTTP servers, some queue consumers (interceptor pattern), GraphQL servers.

**How**
- Order matters: parse body → auth → authorisation → handler; global vs route-scoped middleware.

**Uses**
- Request IDs, metrics, panic recovery, compression, security headers, API key extraction.

## 8) Requests: body, headers, query params

**Short notes**
- **Query:** filter/pagination in URL. **Headers:** metadata (auth, content-type, tracing, caching). **Body:** payload (often JSON); size limits matter.

**Why**
- Correct parsing and limits prevent abuse; caching and auth depend on headers.

**Where**
- Every HTTP handler; multipart for uploads; content negotiation for JSON vs binary.

**How**
- Typed access per framework; max body size; stream large uploads; reject wrong `Content-Type`.

**Uses**
- Search filters (`?q=&page=`), file uploads, idempotency keys in headers, conditional requests (`If-None-Match`).

## 9) Handlers, controllers, services

**Short notes**
- **Handler/controller:** HTTP layer—parse, status codes, thin. **Service/use-case:** business orchestration. **Repositories:** data access (optional separate layer).

**Why**
- Keeps transport details out of domain logic; easier testing and reuse (same service from CLI, worker, HTTP).

**Where**
- Layered or hexagonal architecture in typical web apps and microservices.

**How**
- Controller calls service with validated DTOs; service returns domain result or errors; avoid fat controllers.

**Uses**
- CRUD APIs, complex workflows, sharing logic between REST and background jobs.

## 10) CRUD deep dive

**Short notes**
- **Create / Read / Update / Delete** on resources—maps to POST/GET/PUT|PATCH/DELETE for REST-ish APIs; nuances: partial updates, soft delete, bulk ops.

**Why**
- Foundation of most backends; misunderstandings cause non-idempotent PATCH or leaky deletes.

**Where**
- Admin tools, SaaS APIs, internal entity management.

**How**
- Define resource lifecycle, FK constraints, transactions for multi-row updates, optimistic locking for concurrent edits.

**Uses**
- User profiles, orders, CMS content, configuration records.

## 11) RESTful best practices

**Short notes**
- Resources as nouns, stateless requests, sensible status codes, pagination, versioning strategy, consistent error shape.

**Why**
- Predictable APIs reduce client bugs and ease evolution across teams.

**Where**
- Public HTTP APIs; internal platforms that many services consume.

**How**
- HATEOAS only if valued; hypermedia optional; prefer OpenAPI docs; plural collections; `$filter` conventions or documented query DSL.

**Uses**
- Product APIs, partner integrations, SDK generation, backward-compatible rollouts.

## 12) Databases and persistence

**Short notes**
- Relational vs document vs key-value vs OLAP trade-offs; schemas, migrations, indexes, constraints, transactions.

**Why**
- Data outlives application code; wrong model → performance debt and inconsistent state.

**Where**
- Primary store for domains; read replicas for scale; caches are not authoritative.

**How**
- Migrations tooling; connection pooling; query plans; FK + unique constraints encode invariants alongside app code.

**Uses**
- Orders/ledger, analytics pipelines feeding warehouses, CMS, audit trails.

## 13) Business logic layer (BLL)

**Short notes**
- Rules that encode “what must be true”—pricing, workflows, approvals—not just DB CRUD wrappers.

**Why**
- Prevents scattering rules across controllers and triggers; clarity for compliance and audits.

**Where**
- Domain services/modules; bounded contexts in DDD-ish designs.

**How**
- Pure functions where possible; explicit policy objects; keep side effects orchestrated not hidden.

**Uses**
- Discount engines, entitlement checks, state machines (order shipped → delivered), entitlement upgrades.

## 14) Caching

**Short notes**
- Store copies closer/faster—browser, CDN, reverse proxy, in-process, Redis—with TTL and invalidation story.

**Why**
- Cuts latency and DB load; often largest lever for cost/performance when correct.

**Where**
- Read-heavy endpoints, session stores, rendered fragments, computed aggregates.

**How**
- Cache-aside vs write-through; stampede mitigation; keyed by versioned entity; beware auth-sensitive data.

**Uses**
- Home feeds, config blobs, rate limits, idempotency response memoization (careful).

## 15) Transactional email

**Short notes**
- Outbound mails tied to app events (verification, receipts, resets) usually via ESP (SendGrid, SES, Postmark)—templates + delivery tracking.

**Why**
- Offload deliverability/spam reputations from your app servers; audited customer comms.

**Where**
- After successful transactions; async preferred to avoid slowing HTTP.

**How**
- Queue jobs; idempotent sends; bounce/webhook handling; PII in templates controlled.

**Uses**
- Auth flows, billing notifications, alerts, marketing (with consent) via same or separate pipeline.

## 16) Task queues and scheduling

**Short notes**
- Decouple heavy or slow work: workers consume jobs (Redis, SQS, RabbitMQ); cron/scheduler kicks periodic tasks.

**Why**
- HTTP stays fast; retries with backoff isolate failures.

**Where**
- Emails, image processing, ETL bursts, SLA-friendly fan-out.

**How**
- At-least-once semantics → idempotent workers; DLQs; visibility timeouts; concurrency limits.

**Uses**
- Nightly reconciliation, webhook retries, index rebuilds, invoice generation batch.

## 17) Elasticsearch

**Short notes**
- Search/analytics engine (inverted index, aggregations)—often complementary to relational DB master data.

**Why**
- Full-text relevance, typo tolerance, facets at scale—not every DB does this well.

**Where**
- Product search, log analytics (ELK), autocomplete, investigative queries.

**How**
- Index mappings, analysers; sync pipeline (CDC, dual-write, or periodic); cluster sizing and shards.

**Uses**
- E-commerce catalogue search, observability backends, similarity search pipelines (with vector fields in modern stacks).

## 18) Error handling

**Short notes**
- Map failures to stable error codes/messages; log rich context server-side; never leak secrets in 500 bodies.

**Why**
- Clients need actionable errors; operators need correlation (request ID).

**Where**
- Global exception filters, gRPC status codes, domain-level `Result` types.

**How**
- Typed error taxonomy; retry guidance for 429/503; validation vs auth vs not-found distinction.

**Uses**
- API UX, mobile offline handling, circuit breakers in callers, support triage.

## 19) Configuration management

**Short notes**
- Externalise settings (ports, DB URLs, feature flags)—12-factor style; secrets never in git; per-environment overlays.

**Why**
- Same artefact across envs; rotation without rebuild; fewer “works on my machine” incidents.

**Where**
- Env vars, secret managers (Vault, AWS SM), Kubernetes ConfigMaps/Secrets, `.env` locally only.

**How**
- Typed config load at boot; fail fast on missing required keys; split public vs secret.

**Uses**
- Feature toggles, rate limits, third-party API base URLs, DB pool sizes.

## 20) Logging, monitoring, observability

**Short notes**
- **Logs:** events (structured JSON). **Metrics:** counters/gauges/histograms. **Traces:** request path across services. **SLOs/alerts:** human paging on symptoms.

**Why**
- You cannot fix what you cannot see post-deploy; outages are ambiguous without signals.

**Where**
- APM agents, OpenTelemetry, Prometheus/Grafana, ELK, cloud-native stacks.

**How**
- Correlation IDs; log levels; RED/USE dashboards; graceful sampling under load.

**Uses**
- Latency regressions, error budget tracking, dependency failure diagnosis, audit trails.

## 21) Graceful shutdown

**Short notes**
- On SIGTERM: stop accepting new work, drain in-flight HTTP/queues, flush buffers, close DB pools with timeout.

**Why**
- Prevents clipped requests during deployments and scaling events; avoids duplicate side effects without idempotency.

**Where**
- Container orchestrators (K8s preStop hooks), server runtimes that support shutdown hooks.

**How**
- Health checks flip to unhealthy; wait for drain window; coordinate with load balancer deregistration.

**Uses**
- Zero-downtime releases, autoscaling scale-in, local dev Ctrl+C parity.

## 22) Security

**Short notes**
- OWASP-style risks: injection, broken auth, SSRF, dependency vulns; principle of least privilege; encryption at rest/in transit.

**Why**
- Backends handle secrets and authoritative data—they are prime targets.

**Where**
- Every layer—TLS, CSP for admin UIs served by backend, input validation, supply chain scans.

**How**
- Stay patched; parameterized queries; rate limits; audit logs; secret rotation; SBOM/licensing checks.

**Uses**
- Compliance (GDPR-ish controls), bounty programs baseline, SOC2 preparedness.

## 23) Scaling and performance

**Short notes**
- Vertical vs horizontal scaling; stateless app tiers; offload work; profile before micro-optimizing hot paths.

**Why**
- User growth and spikes demand capacity planning—not accidental architecture.

**Where**
- Load tests, staged rollouts, autoscaling groups, read replicas, edge caching.

**How**
- Measure p95/p99; fix N+1 queries; batch IO; pagination; circuit-breaker timeouts sized to SLAs.

**Uses**
- Black Friday drills, SLA-backed APIs, reducing cloud bill while keeping latency budgets.

## 24) Concurrency and parallelism

**Short notes**
- Many requests in flight (async IO, thread pools) vs doing more CPU work at once; race conditions on shared state.

**Why**
- Throughput and responsiveness depend on model; bugs here are subtle (TOCTOU, lost updates).

**Where**
- Web servers, worker pools, DB transaction isolation, distributed locks when unavoidable.

**How**
- Choose async vs threads per workload; avoid shared mutable state; understand isolation levels.

**Uses**
- High fan-in APIs, chatty DB access with connection limits, background worker throughput tuning.

## 25) Object storage and large files

**Short notes**
- Blobs in S3/GCS/Azure Blob—presigned uploads/downloads, multipart, lifecycle rules; not in app server RAM.

**Why**
- Durability and bandwidth at scale; keeps API processes light and restart-safe.

**Where**
- Media, exports, backups, ML datasets references, user-upload flows.

**How**
- Client → presigned PUT; virus scan worker; metadata row in DB; CDN in front for reads when public.

**Uses**
- Avatars, PDF invoices, video, CSV export downloads, static site artifacts from CI.

## 26) Real-time backend systems

**Short notes**
- Push models: WebSockets, SSE, long polling; often paired with pub/sub (Redis, Kafka) for fan-out.

**Why**
- Live UX (collab, alerts) without aggressive client polling.

**Where**
- Chat, dashboards, games, trading tickers, presence systems.

**How**
- Sticky sessions or shared connection registry; backpressure; auth on upgrade; heartbeats.

**Uses**
- Notifications stream, collaborative editors operational layer, ops live tail UIs.

## 27) Testing and code quality

**Short notes**
- Unit (pure logic), integration (DB/queue wired), contract (consumer-driven), E2E for critical flows; lint/format/static types.

**Why**
- Regressions are expensive live; tests document intent faster than stale wiki pages.

**Where**
- CI on every PR; local pre-push; ephemeral test DB/seeds.

**How**
- Test pyramid reality check; flaky test quarantine policy; deterministic clocks for time logic.

**Uses**
- Refactors with confidence, TDD hotspots (pricing/rules), verifying migrations.

## 28) Twelve-factor app

**Short notes**
- Methodology for modern services: codebase, deps, config, backing services, build/release/run, processes, port binding, concurrency, disposability, dev/prod parity, logs as streams, admin one-off tasks.

**Why**
- Portable, observable, operable deployments—checks many backend hygiene boxes at once.

**Where**
- Cloud-native apps, Kubernetes services, reviews of “is this prod-ready?”.

**How**
- One codebase many deploys; stateless workers; expose via port; export logs to stdout aggregated externally.

**Uses**
- Onboarding checklist, architecture review gate, aligning dev and SRE expectations.

## 29) OpenAPI specification

**Short notes**
- Machine-readable description of HTTP APIs (paths, schemas, examples)—Swagger ecosystem.

**Why**
- Single source fuels docs, mocking, codegen, linting drift detection.

**Where**
- Public API programmes, gateway validation, frontend type generation contracts.

**How**
- Code-first decorators or spec-first YAML; versioning in info block; spectral-style lint CI.

**Uses**
- Client SDK stubs, QA contract tests, partner sandbox alignment, changelog from diff.

## 30) Webhooks

**Short notes**
- Your HTTP callbacks on partner events—or you expose URLs for subscribers; signatures (HMAC) verify origin.

**Why**
- Event-driven integrations without polling; standard pattern for payments, Git platforms, SaaS integrations.

**Where**
- Outbound notifier service; inbound route with replay protection idempotency.

**How**
- Verify signature + timestamp skew; enqueue processing; exponential retries respecting 429 Retry-After; DLQ poisoning alerts.

**Uses**
- Stripe events, Slack apps, SaaS provisioning hooks, CMS publish hooks.

## 31) DevOps for backend engineers

**Short notes**
- CI/CD pipelines, artefacts, infra-as-code familiarity, rollout strategies (blue/green, canary), on-call rotations.

**Why**
- Code is not production until observable, repeatable, reversible deployments exist.

**Where**
- GitHub Actions/Jenkins/GitLab CI, Helm/Terraform summaries, paging policies.

**How**
- Build → test → scan → staged promote; feature flags tie release to rollout; rollback runbooks rehearsed.

**Uses**
- Shipping safely weekly/daily, ownership from laptop to pager, shortening MTTR collaboratively with SRE.